# Đồ án MNIST: MLP tự viết so với MLP của scikit-learn

## 1. Cài đặt và import

In [ ]:
import sys
import time
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier as SklearnMLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# sklearn sẽ cảnh báo chưa hội tụ vì max_iter=20 (chủ ý, để so sánh công bằng với MLP tự viết)
warnings.filterwarnings("ignore", category=Warning, module="sklearn")

sys.path.append(os.path.abspath('..'))
from src.data import load_prepared_mnist
SEED = 42

## 2. Nạp data MNIST

In [ ]:
X_train_full, y_train_raw, X_test_full, y_test_all = load_prepared_mnist()

# Đảm bảo kiểu dữ liệu và hình dạng phù hợp; không thay đổi giá trị đã tiền xử lý.
X_all = np.asarray(X_train_full, dtype=np.float32)
y_all = np.asarray(y_train_raw).reshape(-1)
X_test_full = np.asarray(X_test_full, dtype=np.float32)
y_test_all = np.asarray(y_test_all).reshape(-1)

if X_all.ndim != 2 or X_all.shape[1] != 784:
    raise ValueError(f"X_train phải có shape (N, 784) sau preprocess; nhận được {X_all.shape}")
if X_test_full.ndim != 2 or X_test_full.shape[1] != 784:
    raise ValueError(f"X_test phải có shape (N, 784) sau preprocess; nhận được {X_test_full.shape}")
if len(X_all) != len(y_all) or len(X_test_full) != len(y_test_all):
    raise ValueError("Số mẫu trong X và y không khớp.")

print("Train gốc:", X_all.shape, y_all.shape, X_all.dtype)
print("Test gốc :", X_test_full.shape, y_test_all.shape, X_test_full.dtype)
print("Khoảng pixel:", float(X_all.min()), float(X_all.max()))
print("Các nhãn trong train:", np.unique(y_all))

## 3. Chia tập dữ liệu thành train và validation

In [ ]:
TRAIN_LIMIT = None
VALIDATION_LIMIT = None
TEST_LIMIT = None

def stratified_subsample(X, y, limit, random_state=None):
    """Lấy mẫu phân tầng từ X, y với số lượng mẫu tối đa là limit."""
    if limit is None or limit >= len(y):
        return X.copy(), y.copy()
    if limit < len(np.unique(y)):
        raise ValueError("limit phải ít nhất bằng số lớp để nhãn nào cũng có khả năng xuất hiện")

    X_keep, _, y_keep, _ = train_test_split(
        X, y, train_size=limit, stratify=y, random_state=random_state
    )
    return X_keep, y_keep

# Chia từ X_all, y_all (không ghi đè y_all) nên chạy lại cell này nhiều lần vẫn đúng
X_train_all, X_val_all, y_train_all, y_val_all = train_test_split(
    X_all, y_all, test_size=0.1, stratify=y_all, random_state=SEED
)

X_train, y_train = stratified_subsample(X_train_all, y_train_all, TRAIN_LIMIT, random_state=SEED)
X_val, y_val = stratified_subsample(X_val_all, y_val_all, VALIDATION_LIMIT, random_state=SEED)
X_test, y_test = stratified_subsample(X_test_full, y_test_all, TEST_LIMIT, random_state=SEED)

In [ ]:
from IPython.display import display, Markdown

def show(df: pd.DataFrame, caption: str | None = None) -> None:
    if caption:
        display(Markdown(f"**{caption}**"))
    display(df)

over_view = pd.DataFrame([
    ["Train", len(X_train), X_train.shape[1], "Huấn luyện mô hình"],
    ["Validation", len(X_val), X_val.shape[1], "Chọn cấu hình / siêu tham số"],
    ["Test", len(X_test), X_test.shape[1], "Đánh giá cuối cùng"]
], columns=["Tập", "Số mẫu", "Số đặc trưng", "Mục đích"])
show(over_view, "Các tập dữ liệu")

counts_tr  = np.bincount(y_train, minlength=10)
counts_val = np.bincount(y_val,   minlength=10)
counts_te  = np.bincount(y_test,  minlength=10)

fig, ax = plt.subplots(figsize=(11, 4))

x = np.arange(10)
w = 0.27          # độ rộng mỗi cột (3 cột x 0.27 = 0.81 < 1 nên có khoảng trống)

# 3 cột: trái = train, giữa = validation, phải = test
ax.bar(x - w, counts_tr,  w, color="steelblue", label="train")
ax.bar(x,     counts_val, w, color="orange",    label="validation")
ax.bar(x + w, counts_te,  w, color="gray",      label="test")

# Ghi số liệu trên đầu mỗi cột
for xi, (a, b, c) in enumerate(zip(counts_tr, counts_val, counts_te)):
    ax.text(xi - w, a + 60, f"{a:,}", ha="center", fontsize=6, rotation=90)
    ax.text(xi,     b + 60, f"{b:,}", ha="center", fontsize=6, rotation=90)
    ax.text(xi + w, c + 60, f"{c:,}", ha="center", fontsize=6, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(range(10))
ax.set_xlabel("Chữ số")
ax.set_ylabel("Số mẫu")

# Trục Y phải đủ cao cho cột lớn nhất trong 3 tập
ax.set_ylim(0, max(counts_tr.max(), counts_val.max(), counts_te.max()) * 1.22)

ax.set_title("Hình 1 — Phân bố nhãn gần cân bằng giữa 10 lớp và giữa train/val/test")
ax.legend()
fig.tight_layout()
plt.show()

## 4. MLP đã viết bằng tay

In [ ]:
from src.models.mlp import MLPClassifier
print("MLPClassifier được import:", MLPClassifier)
print("Kích thước các lớp ẩn mặc định:", MLPClassifier().hidden_units)

## 5. Các hàm đánh giá

In [ ]:
def evaluate_from_pred(name, y_true, y_pred, fit_s=None, predict_s=None):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "fit_seconds": fit_s,
        "predict_seconds": predict_s,
    }

def timed_fit(model, X, y):
    """Tính thời gian train model"""
    start = time.perf_counter()
    model.fit(X, y)
    times = time.perf_counter() - start
    return model, times

def count_parameter_manual(model):
    """Tính tổng số tham số của model (trọng số + bias).
    Tổng tham số thể hiện dung lượng bộ nhớ của mô hình.
    Quá lớn dễ bị overfitting, quá nhỏ dễ bị underfitting."""
    return int(sum(W.size for W in model.weights_)) + int(sum(b.size for b in model.biases_))

## 6. So sánh MLP tự viết với MLP của scikit-learn

In [ ]:
BASE_HIDDEN = (256, 128)
BASE_LR = 0.08
BASE_EPOCHS = 20
BASE_BATCH = 256
BASE_L2 = 1e-4

manual = MLPClassifier(
    hidden_units=BASE_HIDDEN, learning_rate=BASE_LR,
    epochs=BASE_EPOCHS, batch_size=BASE_BATCH, l2=BASE_L2, seed=SEED
)
manual, manual_fit_s = timed_fit(manual, X_train, y_train)

manual_pred_start = time.perf_counter()
manual_val_pred = manual.predict(X_val)
manual_pred_s = time.perf_counter() - manual_pred_start

sk_mlp = SklearnMLPClassifier(
    hidden_layer_sizes=BASE_HIDDEN,
    activation="relu",
    solver="sgd",
    alpha=BASE_L2,
    batch_size=BASE_BATCH,
    learning_rate="constant",
    learning_rate_init=BASE_LR,
    max_iter=BASE_EPOCHS,
    shuffle=True,
    momentum=0.0,
    nesterovs_momentum=False,
    tol=0.0,
    n_iter_no_change=BASE_EPOCHS + 1,
    random_state=SEED,
    verbose=False,
)
sk_mlp, sk_fit_s = timed_fit(sk_mlp, X_train, y_train)
sk_pred_start = time.perf_counter()
sk_val_pred = sk_mlp.predict(X_val)
sk_pred_s = time.perf_counter() - sk_pred_start

sk_n_params = int(sum(coef.size for coef in sk_mlp.coefs_) + sum(inter.size for inter in sk_mlp.intercepts_))

baseline_results = pd.DataFrame([
    evaluate_from_pred("MLP tự viết",     y_val, manual_val_pred, manual_fit_s, manual_pred_s),
    evaluate_from_pred("MLP scikit-learn", y_val, sk_val_pred,     sk_fit_s,     sk_pred_s),
])
baseline_results["n_parameters"] = [count_parameter_manual(manual), sk_n_params]
display(baseline_results.style.format({
    "accuracy": "{:.4f}", "precision_macro": "{:.4f}",
    "recall_macro": "{:.4f}", "f1_macro": "{:.4f}",
    "fit_seconds": "{:.4f}", "predict_seconds": "{:.4f}",
    "n_parameters": "{:,}"
}))
print("Số epoch thực tế của sklearn:", sk_mlp.n_iter_)
print("Số tham số MLP tự viết:", count_parameter_manual(manual))
print("Số tham số MLP sklearn:", sk_n_params)

## 7. Biểu đồ so sánh metric và thời gian

In [ ]:
metrics_to_plot = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
baseline_results.set_index("model")[metrics_to_plot].plot(kind="bar", figsize=(10, 5), ylim=(0.9, 1.0))
plt.title("So sánh chất lượng dự đoán trên validation")
plt.ylabel("Điểm số")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

baseline_results.set_index("model")[["fit_seconds", "predict_seconds"]].plot(kind="bar", figsize=(9, 4))
plt.title("Thời gian chạy trên cùng tập dữ liệu")
plt.ylabel("Giây")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Loss curve

Lưu ý: `loss_curve_` của sklearn đã cộng thêm số hạng phạt L2, nên giá trị loss của hai mô hình
có thể lệch nhau một chút do cách tính, không hoàn toàn so sánh trực tiếp được. Hình dạng đường cong (giảm theo epoch) mới là điều đáng so sánh.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(manual.history_) + 1), manual.history_, marker="o", label="MLP tự viết: train loss")
if hasattr(sk_mlp, "loss_curve_"):
    plt.plot(np.arange(1, len(sk_mlp.loss_curve_) + 1), sk_mlp.loss_curve_,
             marker="s", label="MLP sklearn: loss nội bộ")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Đường cong loss theo epoch")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Confusion matrix và hiệu năng theo từng lớp

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# MLP tự viết
ConfusionMatrixDisplay.from_predictions(
    y_val, manual_val_pred, ax=axes[0],
    cmap="Blues", colorbar=False, values_format="d"
)
axes[0].set_title("MLP tự viết (Validation)", fontsize=12, pad=10)

# MLP scikit-learn
ConfusionMatrixDisplay.from_predictions(
    y_val, sk_val_pred, ax=axes[1],
    cmap="Blues", colorbar=False, values_format="d"
)
axes[1].set_title("MLP scikit-learn (Validation)", fontsize=12, pad=10)

fig.suptitle("So sánh ma trận nhầm lẫn trên tập Validation", fontsize=14, fontweight="bold", y=1.05)

plt.tight_layout()
plt.show()

# In báo cáo chi tiết
print("=" * 60)
print("MLP tự viết — Classification Report")
print(classification_report(y_val, manual_val_pred, digits=4, zero_division=0))

print("=" * 60)
print("MLP scikit-learn — Classification Report")
print(classification_report(y_val, sk_val_pred, digits=4, zero_division=0))

## 10. Thí nghiệm siêu tham số — learning rate

In [ ]:
LR_VALUES = [0.001, 0.01, 0.05, 0.08, 0.1]
lr_row = []
lr_models = {}
for lr_value in LR_VALUES:
    model = MLPClassifier(
        hidden_units=BASE_HIDDEN,
        learning_rate=lr_value,
        epochs=BASE_EPOCHS,
        batch_size=BASE_BATCH,
        l2=BASE_L2,
        seed=SEED
    )

    model, time_train = timed_fit(model, X_train, y_train)

    time_val_start = time.perf_counter()
    y_val_predict = model.predict(X_val)
    time_val = time.perf_counter() - time_val_start

    evaluate = evaluate_from_pred("MLP tự viết", y_val, y_val_predict, time_train, time_val)
    evaluate["learning rate"] = lr_value
    evaluate["train_loss_cuoi"] = model.history_[-1]

    lr_row.append(evaluate)
    lr_models[lr_value] = model  # dùng để lấy history_ vẽ loss

lr_results = pd.DataFrame(lr_row)
display(lr_results)

plt.figure(figsize=(12, 5))
x = lr_results["learning rate"]

plt.plot(x, lr_results["accuracy"], label="Accuracy", color="blue", marker="o")
plt.plot(x, lr_results["precision_macro"], label="Precision Macro", color="red", marker="s")
plt.plot(x, lr_results["recall_macro"], label="Recall Macro", color="green", marker="x")
plt.plot(x, lr_results["f1_macro"], label="F1 Macro", color="gray", marker="D")

plt.xscale("log")
plt.xlabel("Learning Rate (thang log)")
plt.ylabel("Độ đo")
plt.title("Performance Metrics theo Learning Rate")
plt.legend()
plt.grid(True)
plt.show()

# Biểu đồ so sánh Train Loss theo Epoch
plt.figure(figsize=(9, 5))
for learning_rate, model in lr_models.items():
    plt.plot(
        range(1, len(model.history_) + 1),
        model.history_,
        label=f"LR = {learning_rate}"
    )

plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.title("Train Loss theo Learning Rate")
plt.legend()
plt.grid(True)
plt.show()

## 11. Thí nghiệm kiến trúc mạng

In [ ]:
architectures = [(128,), (256,), (128, 64), (256, 128), (256, 128, 64)]
ket_qua = []
dict_model = {}

for hidden in architectures:
    model = MLPClassifier(
        hidden_units=hidden,
        learning_rate=BASE_LR,
        epochs=BASE_EPOCHS,
        batch_size=BASE_BATCH,
        l2=BASE_L2,
        seed=SEED
    )

    model, fit_s = timed_fit(model, X_train, y_train)

    pred_start = time.perf_counter()
    pred = model.predict(X_val)
    pred_s = time.perf_counter() - pred_start

    ev = evaluate_from_pred("MLP tự viết", y_val, pred, fit_s, pred_s)
    ev["hidden_units"] = str(hidden)
    ev["train_loss_cuoi"] = model.history_[-1]
    ket_qua.append(ev)
    dict_model[hidden] = model  # giữ model để vẽ loss

arch_df = pd.DataFrame(ket_qua)
display(arch_df)

x = arch_df["hidden_units"]
plt.figure(figsize=(12, 5))
plt.plot(x, arch_df["accuracy"],        label="Accuracy",        color="blue",  marker="o")
plt.plot(x, arch_df["precision_macro"], label="Precision Macro", color="red",   marker="s")
plt.plot(x, arch_df["recall_macro"],    label="Recall Macro",    color="green", marker="x")
plt.plot(x, arch_df["f1_macro"],        label="F1 Macro",        color="gray",  marker="D")
plt.xlabel("Hidden Units")
plt.ylabel("Độ đo")
plt.title("Performance Metrics theo Hidden Units")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Loss curve
if "dict_model" not in globals() or not dict_model:
    raise RuntimeError(
        "dict_model chưa có dữ liệu — chạy cell 'Thí nghiệm kiến trúc mạng' phía trên trước."
    )

plt.figure(figsize=(9, 5))
for hidden, model in dict_model.items():
    history = np.asarray(getattr(model, "history_", model), dtype=float)
    plt.plot(
        range(1, history.size + 1),
        history,
        label=f"Hidden = {hidden}"
    )
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.title("Train Loss theo cấu hình Hidden Units")
plt.legend()
plt.grid(True)
plt.show()

## 12. Thí nghiệm theo Epoch

In [ ]:
EPOCH_VALUES = [2, 5, 10, 15, 20]
epoch_rows = []
epoch_models = {}

for epochs in EPOCH_VALUES:
    model = MLPClassifier(
        hidden_units=BASE_HIDDEN, learning_rate=BASE_LR,
        epochs=epochs, batch_size=BASE_BATCH, l2=BASE_L2, seed=SEED
    )
    model, fit_s = timed_fit(model, X_train, y_train)

    time_start_pred = time.perf_counter()
    pred = model.predict(X_val)
    pred_s = time.perf_counter() - time_start_pred

    row = evaluate_from_pred("MLP tự viết", y_val, pred, fit_s, pred_s)
    row["epochs"] = epochs
    row["final_train_loss"] = model.history_[-1]
    epoch_rows.append(row)
    epoch_models[epochs] = model  # ghi lại để tính loss

epoch_results = pd.DataFrame(epoch_rows)
display(epoch_results[["epochs", "accuracy", "f1_macro", "final_train_loss", "fit_seconds"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(epoch_results["epochs"], epoch_results["accuracy"], marker="o", label="Accuracy")
axes[0].plot(epoch_results["epochs"], epoch_results["f1_macro"], marker="s", label="F1-macro")
axes[0].plot(epoch_results["epochs"], epoch_results["precision_macro"], marker="D", label="Precision")
axes[0].plot(epoch_results["epochs"], epoch_results["recall_macro"], marker="p", label="Recall")

axes[0].set_xlabel("Số epoch")
axes[0].set_ylabel("Điểm số")
axes[0].set_ylim(0.9, 1.0)
axes[0].set_title("Chất lượng validation theo số epoch")
axes[0].legend()

axes[1].plot(epoch_results["epochs"], epoch_results["final_train_loss"], marker="o")
axes[1].set_xlabel("Số epoch")
axes[1].set_ylabel("Train loss cuối")
axes[1].set_title("Train loss theo số epoch")
plt.tight_layout()
plt.show()

## 13. Chọn cấu hình tốt nhất bằng validation và đánh giá trên test

In [ ]:
# Predict lại toàn bộ, gom kết quả vào candidate_rows, rồi sắp theo F1, Accuracy.
# Mỗi cấu hình được mô tả bằng (hidden, lr, epochs) để loại các cấu hình trùng nhau
# (cấu hình baseline xuất hiện ở cả 3 thí nghiệm).
candidate_rows = []

def add_candidate(source, model, hidden, lr_value, epochs):
    pred = model.predict(X_val)
    candidate_rows.append({
        "source": source,
        "configuration": f"hidden={hidden}, lr={lr_value}, epochs={epochs}",
        "model": model,
        "f1_macro": f1_score(y_val, pred, average="macro"),
        "accuracy": accuracy_score(y_val, pred),
    })

for hidden, model in dict_model.items():
    add_candidate("architecture", model, hidden, BASE_LR, BASE_EPOCHS)
for epochs, model in epoch_models.items():
    add_candidate("epochs", model, BASE_HIDDEN, BASE_LR, epochs)
for lr_value, model in lr_models.items():
    add_candidate("learning_rate", model, BASE_HIDDEN, lr_value, BASE_EPOCHS)

candidate_table = (
    pd.DataFrame(candidate_rows)
    .drop_duplicates(subset="configuration", keep="first")
    .reset_index(drop=True)
)
display(candidate_table.drop(columns="model").sort_values("f1_macro", ascending=False).head(10).round(4))

best_row = candidate_table.sort_values(
    ["f1_macro", "accuracy"], ascending=False
).iloc[0]
best_manual = best_row["model"]
print("Cấu hình MLP tự viết được chọn từ validation:", best_row["configuration"])
print(f"Validation F1-macro={best_row['f1_macro']:.4f}, accuracy={best_row['accuracy']:.4f}")

# Đánh giá test chỉ sau bước chọn cấu hình
best_test_pred = best_manual.predict(X_test)
test_result = {
    "model": "MLP tự viết (được chọn bằng validation)",
    "configuration": best_row["configuration"],
    "accuracy": accuracy_score(y_test, best_test_pred),
    "precision_macro": precision_score(y_test, best_test_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, best_test_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, best_test_pred, average="macro", zero_division=0),
}
test_results = pd.DataFrame([test_result])
display(test_results.round(4))

ConfusionMatrixDisplay.from_predictions(y_test, best_test_pred)
plt.title("Confusion matrix trên tập Test")
plt.show()
print(classification_report(y_test, best_test_pred, digits=4, zero_division=0))